# Parts 2–3 · Check and run
Use the verified bundle from Part 1. Inspect assumptions before running. There is no API key or external model call. The no-events scenario is a software control, not an untreated patient.

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util

# In Colab, the public GitHub version must already contain this release.
INSTALL_PACKAGE = True
REPO_URL = "https://github.com/hylixinsights/VirtualTissue-scRNAseq.git"
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((d for d in candidates if (d / "pyproject.toml").exists() and (d / "src/vtsc").exists()), None)
if ROOT is None:
    ROOT = Path("/content/VirtualTissue-scRNAseq") if IN_COLAB else Path.home() / "VirtualTissue-scRNAseq"
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
if INSTALL_PACKAGE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT) + "[notebooks]"], check=True)
sys.path.insert(0, str(ROOT / "src"))
print("Code:", ROOT)
# Leave WORK below local, or explicitly mount Drive and choose a Drive path.
WORK = Path(os.environ.get("VTSC_NOTEBOOK_WORK", str(ROOT / "outputs/notebook_demo")))
WORK.mkdir(parents=True, exist_ok=True)
print("Files:", WORK)


In [ ]:
BUNDLE = WORK / 'input.zip'  # change to the exact ZIP from Part 1
RUN_NAME = 'baseline'
HOURS = 48.0
SEED = 21
INTERVAL_HOURS = 1.0
MAX_CELLS = 5000
MAX_EVENTS = 50000
MODEL_OVERRIDE = None  # Path('/path/to/my_model.yaml')
RUN_CONTROL = True


In [ ]:
from vtsc.inputs import read_bundle
from vtsc.model import load_model, mean_field
cells, embedded, provenance = read_bundle(BUNDLE)
model = load_model(MODEL_OVERRIDE) if MODEL_OVERRIDE else embedded
initial = [sum(c['state'] == s for c in cells) for s in model['states']]
print('Cells:', len(cells), 'states:', dict(zip(model['states'], initial)))
print('Assumptions:', model)
print('Uncapped expected final population:', float(mean_field(initial, [HOURS], model)[0].sum()))
print('Source:', provenance)


## Execute the software experiment
Running the next cell starts the simulation. Existing output names fail instead of overwriting previous runs. Inspect completion status; increase caps only intentionally.

In [ ]:
from vtsc.results import run_bundle, compare_runs
result = run_bundle(BUNDLE, WORK / RUN_NAME, model_path=MODEL_OVERRIDE, hours=HOURS, seed=SEED,
                    interval=INTERVAL_HOURS, max_cells=MAX_CELLS, max_events=MAX_EVENTS)
if RUN_CONTROL:
    control = run_bundle(BUNDLE, WORK / (RUN_NAME + '_control'), model_path=MODEL_OVERRIDE, scenario='no-events',
                         hours=HOURS, seed=SEED, interval=INTERVAL_HOURS, max_cells=MAX_CELLS, max_events=MAX_EVENTS)
    assert control['events'] == 0 and control['initial_cells'] == control['final_cells']
    compare_runs(WORK / RUN_NAME, WORK / (RUN_NAME + '_control'), WORK / (RUN_NAME + '_comparison'))
result


In [ ]:
from IPython.display import display, Image
print('Results ZIP:', result['results_zip'])
display(Image(filename=str(WORK / RUN_NAME / 'overview.png')))
